# connexplorer: every data structure, and a worked Dm9 analysis

`connexplorer` gives one API over the FlyWire FAFB (v783) and Male CNS (v1.0) connectomes. This notebook is in two parts.

**Part 1** goes through each object the package exposes, in the order you meet them: what it holds, where its data comes from on disk, and what you can ask of it.

| object | holds | comes from |
|---|---|---|
| `Dataset` | manifest, cached tables, id lookups | `data/<dataset>/tables/manifest.json` and the Parquet tables |
| `NeuronSet` / `Neuron` | a set of cells (dense ids) | any selection over `ds.cells` |
| `Connectivity` | CSR/CSC of the edge table, partner tables, cached totals | `edges.parquet`, built once in memory |
| `ConnBlock` | one pre-set × post-set sub-matrix | a slice of the CSR |
| `TypeMatrix` / `TypeBlock` | type-to-type totals | `type_matrix.npz` |
| `Synapses` | lazy access to 80 M synapse locations | `synapses.parquet` and its post-sorted twin, never loaded whole |
| `SkeletonStore` | SWC skeletons as navis neurons | a zip or folder under `tables/skeletons/`, or fetched per body |
| `Compartments` / `Cable` | a skeleton split for cable modelling, and the passive model | computed from a skeleton |
| `Comparison` | two sets' partner profiles aligned across datasets | the vendor type map on `ds.types` |
| viewer builders | Neuroglancer / Codex URLs | per-dataset configuration |

**Part 2** uses those objects to redo the connectivity half of a real analysis: how the right-hemisphere Dm9 cells wire R7 and R8 photoreceptors to each other across visual columns.

Every table you get back is a polars DataFrame; root ids are the public identifiers; nothing opens a browser.

In [ ]:
import numpy as np
import polars as pl
import scipy.sparse as sp
import plotly.graph_objects as go
import plotly.express as px

import connexplorer as cnx
print(cnx.__version__)

# Part 1: the data structures

## 1. `Dataset`

`cnx.open` takes a dataset name or a directory. By name it searches `cnx.config.data_dir`, then `$CONNEXPLORER_DATA`, then `./data`, for a folder whose `tables/manifest.json` names that dataset, and picks the latest version. Opening reads only the manifest; nothing else is touched until you ask for it.

In [ ]:
ds = cnx.open("flywire")
print(ds)
print("tables on disk:", sorted(ds.manifest.tables))
print("provenance:", {k: v["path"].rsplit("/", 1)[-1] for k, v in ds.manifest.source_files.items()})
ds.info()

### Tables are cached polars frames

`ds.cells`, `ds.types`, `ds.edges`, `ds.edges_by_neuropil` and `ds.columns` load on first access and stay in memory. They are the frames themselves, not copies, which is safe because polars frames are immutable.

Two id systems live side by side:

- **root ids** (`root_id`, int64) are the vendor identifiers and the only ids the public API takes or returns;
- **dense ids** (`id`, 0..N-1) are row positions in `ds.cells` and the axes of every matrix. Cells are stored sorted by type, so each type occupies a contiguous range of dense ids, which is what makes type blocks slices instead of gathers.

In [ ]:
print(ds.cells.shape, ds.cells.columns)
display(ds.cells.head(3))
print(ds.types.shape, ds.types.columns)
display(ds.types.filter(pl.col("nt_source") == "literature").select("type", "n_cells", "nt", "nt_predicted", "nt_source").head(5))
display(ds.edges.head(3))
print("edges_by_neuropil:", ds.edges_by_neuropil.shape, "| columns table:", ds.columns.shape, ds.columns.columns)

In [ ]:
# id lookups, both directions
rid = 720575940599755718
idx = ds.idx_of([rid])                 # root ids -> dense ids (vectorized searchsorted)
print("dense id", idx, "-> root id", ds.root_ids[idx])
lo, hi = ds._type_ranges["T4a"]        # types are contiguous dense-id ranges
print(f"T4a occupies dense ids {lo}..{hi - 1} ({hi - lo} cells); ds.type_idx('T4a') gives that arange")

## 2. `NeuronSet` and `Neuron`

Indexing a dataset gives you a `NeuronSet`, a sorted array of dense ids plus a label. A `Neuron` is a `NeuronSet` of size one with scalar conveniences. Every query method below exists on both, and sets support `&`, `|`, `-`, positional indexing and iteration.

Ways to make one:

In [ ]:
n     = ds[rid]                              # one root id -> Neuron
t4a   = ds["T4a"]                            # one type -> NeuronSet
t4    = ds[["T4a", "T4b", "T4c", "T4d"]]     # union of types
some  = ds[[rid, 720575940632504874]]        # list of root ids
right = ds.select(type="T4a", side="R")      # column == value filters
gaba  = ds.select(pl.col("nt") == "GABA")    # any polars expression over ds.cells
print(n, t4a, t4, some, right, gaba, sep="\n")
print("scalars on a Neuron:", n.root_id, n.id, n.type, n.side, n.nt)
print("arrays on a set:", t4a.ids[:3], t4a.idx[:3], t4a.types)
print("algebra:", len(t4 | gaba), len(t4a & right), len(t4a - right), right[0], len(right[:10]))
n.cell

`grp.cells` is the matching slice of `ds.cells`; `grp.degree()` gives per-cell partner counts and synapse totals (whole dataset, or `within=True` for the induced subgraph); `hubs`, `reciprocal`, `degree_distribution` and `summary` build on that.

In [ ]:
display(right.cells.head(3))
display(right.degree().head(3))
display(right.hubs(k=3, by="out_syn"))
print(right.summary())

## 3. `Connectivity`

`ds.connectivity` builds a CSR matrix (rows = pre, cols = post, dense-id order) from the edge table on first use, plus a CSC copy for input-side slicing. Both are cached. Self-connections (autapses) exist in the data and are masked by default; flip `ds.connectivity.autapses = True` and every result, including the type matrix diagonal, includes them.

Whole-dataset per-cell and per-type totals (`cell_totals`, `type_totals`) are cached per autapse setting and serve as normalization denominators.

In [ ]:
conn = ds.connectivity
M = conn.sparse                        # scipy CSR, N x N
print(type(M).__name__, M.shape, f"{M.nnz:,} pairs, {int(M.sum()):,} synapses (autapses masked)")
print("row of one cell = its outputs:", M.indptr[n.id + 1] - M.indptr[n.id], "partners")
tot = conn.cell_totals
print("totals for that cell: out_syn", tot.out_syn[n.id], "in_syn", tot.in_syn[n.id], "out_deg", tot.out_deg[n.id])
conn.autapses = True
print("with autapses:", f"{conn.sparse.sum():,}", "synapses")
conn.autapses = False

### Partner tables

`outputs()` / `inputs()` on any set return one row per partner cell with type, side and synapse count, strongest first. `min_syn` thresholds the per-partner total (never a neuropil fragment). `by` groups by `type`, `side`, `neuropil` or a combination. `normalize=True` adds the set-side fraction, the partner-side whole-dataset fraction and their geometric mean.

In [ ]:
display(n.outputs(min_syn=5).head())
display(n.inputs(by="type").head())
display(n.inputs(by=("type", "neuropil")).head())
display(t4a.inputs(by="type", normalize=True).head())
n.partners().head()

## 4. `ConnBlock`

`ds.connectivity[pre, post]` accepts type names, lists of types, NeuronSets or root ids on each side and returns a block object. Nothing is materialized until you ask for a view: `.sparse` (CSR sub-matrix), `.values` (dense numpy), `.long` (pre, post, n_syn rows), `.frame` (wide polars), `.normalized` (long with fractions), plus `row_ids`, `col_ids`, `row_types`, `col_types`, `sum()`.

In [ ]:
blk = ds.connectivity["T4a", "LPi14"]
print(blk, blk.shape, blk.sparse.nnz, "nonzero pairs")
print("dense:", blk.values.shape, blk.values.dtype, "| rows are T4a root ids:", blk.row_ids[:2], "| cols:", blk.col_ids)
display(blk.long.head(3))
display(blk.normalized.head(3))
print(ds.connectivity[right, "LPi14"].sum(), ds.connectivity[[rid], "LPi14"].values)

## 5. `TypeMatrix` and `TypeBlock`

`ds.connectivity.types` wraps the precomputed type-to-type matrix (`types.parquet` order). Two names give an integer; two lists give a `TypeBlock` with the same views as a cell block. `.normalized` on the whole matrix gives every type pair with `frac_output`, `frac_input` and `weight_norm`, which is what shayan's `aggregate_by_type_normalized` computed, without the half-second recompute.

In [ ]:
tm = ds.connectivity.types
print(tm, "| Mi1 -> T4a:", tm["Mi1", "T4a"])
tb = tm[["Mi1", "Tm3", "Mi9"], ["T4a", "T4b", "T4c", "T4d"]]
display(tb.frame)
display(tm.normalized.filter(pl.col("post_type") == "T4a").sort("n_syn", descending=True).head(5))

## 6. `Synapses`

`synapses.parquet` (80 M rows, dense ids, nanometres) is sorted by presynaptic cell in 1 M-row row groups; `synapses_by_post.parquet` is the same rows sorted by postsynaptic cell. `ds.synapses` never loads either: a filter on the sort column lets polars read only the row groups that can contain the cell, so `n.synapses()` returns in about 2 ms with root ids substituted back in. `ds.synapses.scan()` hands you the LazyFrame for bulk work.

In [ ]:
s_out, s_in = n.synapses("out"), n.synapses("in")
print(s_out.height, "output and", s_in.height, "input synapses of", n)
display(s_in.head(3))
print("to one partner type:", n.synapses(partner="LPi14").height, "| between two sets:", ds.synapses.between(t4a[:20], "LPi14").height)
print("as an array (nm):", cnx.xyz(s_in)[:2], "| lazy scan:", ds.synapses.scan().select(pl.len()).collect().item(), "rows on disk")

## 7. `SkeletonStore`, `Compartments`, `Cable`

`ds.skeletons` finds SWC files under `tables/skeletons/` (one zip or a folder) and parses one straight out of the zip in about 10 ms, always in micrometres. For the Male CNS, where the full set is impractically large, `mc[body].skeleton()` fetches that body from the vendor store and keeps it as SWC.

`cnx.morph.segment` splits a skeleton into compartments (one per linear segment, short ones merged), and `cnx.models.Cable` is the passive model on top. `comp.nearest(xyz_um)` maps points such as synapses to compartments. The example is CT1, the giant cell that contacts every T4/T5 column.

In [ ]:
import navis

ct1 = ds.select(type="CT1", side="R")[0]
sk = ct1.skeleton()
print(ds.skeletons, "|", ct1, "|", sk.n_nodes, "nodes,", f"{float(sk.cable_length) / 1e3:.1f} mm cable,", sk.units)
comp = cnx.morph.segment(sk, min_length_um=2.0)
print(comp); display(comp.table.head(3))
m = cnx.models.Cable(comp, Rm=8000, Ra=400, Cm=0.6)
V = m.steady_state({0: 10e-12})
print(f"10 pA at the root: {V[0]:.3f} mV there, input resistance {m.input_resistance(0):.3f} GOhm")
# where do CT1's T4a input synapses sit on the tree?
syn = ct1.synapses("in", partner="T4a")
comps = comp.nearest(cnx.xyz(syn) / 1e3)
print(syn.height, "T4a -> CT1 synapses land on", len(np.unique(comps)), "compartments")

In [ ]:
sk_small = navis.downsample_neuron(sk, 25, inplace=False)
fig = navis.plot3d(sk_small, backend="plotly", inline=False, color="#999999")
for df, name, color in ((ct1.synapses("in"), "inputs (CT1 is post)", "#d62728"), (ct1.synapses("out"), "outputs (CT1 is pre)", "#1f77b4")):
    p = cnx.xyz(df.sample(3000, seed=0)) / 1e3
    fig.add_trace(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers", marker=dict(size=1.5, color=color), name=name))
fig.update_layout(title="CT1 (right) with a sample of its synapses, micrometres", scene=dict(aspectmode="data"), height=600)
fig.show()

## 8. Viewers and the second dataset

`view()` on any set returns a Neuroglancer URL for spelunker, cave, the FlyWire production viewer or Codex; `partners` adds the top partner types as layers and `synapses=True` their points. `comp.view(ds)` marks compartment centres.

`cnx.open("mcns")` opens the Male CNS with the same API. Its types table carries the vendor's FlyWire name per type, so `cnx.compare` aligns profiles across the two brains and `mc.types_like` translates names.

In [ ]:
print(n.view(partners="in", top=3, synapses=True)[:100], "...")
print(t4a[:3].view(viewer="codex"))
mc = cnx.open("mcns")
print(mc, "| R7 in Male CNS naming:", mc.types_like("R7"))
display(cnx.compare(ds["T4a"], mc["T4a"]).inputs().head(6))
mi1 = mc["Mi1"][0]
print(mi1, "->", mi1.skeleton().n_nodes, "nodes fetched on demand")

# Part 2: Dm9 connectivity in the right optic lobe

Dm9 is a wide-field medulla cell. Each one receives histaminergic input from R7 and R8 photoreceptors across a patch of ommatidia and feeds glutamatergic output back onto R7/R8 across roughly the same patch, so photoreceptors inhibit one another laterally through Dm9. The questions the connectivity alone can answer:

1. What does the Dm9 population look like and who are its partners?
2. For every R7/R8 pair, how many Dm9s link them (the two-hop pathway matrix)?
3. How large are each Dm9's receptive field (RF, cells it reads) and projective field (PF, cells it writes), and how much do they overlap?
4. On the hexagonal column grid, how does the two-hop coupling fall off with distance, and what does the offset kernel look like?
5. Which columns are covered by how many Dm9s, and where do RF and PF differ?
6. Do R7 and R8 in the same column also talk to each other directly?

## 2.1 The population and its partners

In [ ]:
dm9 = ds.select(type="Dm9", side="R")
print(dm9, "| all Dm9:", len(ds["Dm9"]))
display(dm9.inputs(by="type", normalize=True).head(8))
display(dm9.outputs(by="type", normalize=True).head(8))
# per-cell view of the same thing: R7+R8 share of each Dm9's input
per_cell = pl.DataFrame({
    "root_id": dm9.ids,
    "frac_photoreceptor_input": [float(c.inputs(by="type").filter(pl.col("type").is_in(["R7", "R8"]))["frac_input"].sum()) for c in dm9],
})
print(per_cell["frac_photoreceptor_input"].describe())

## 2.2 Photoreceptors on the column grid

The column table assigns R7 and R8 cells (and the columnar medulla types) to hexagonal visual columns with axial coordinates (p, q). We take the right-eye photoreceptors that have a column assignment; that is the population whose two-hop coupling can be placed in space.

In [ ]:
cols = ds.columns.join(ds.cells.select("id", "root_id", "type"), on="id")
pr_cols = cols.filter(pl.col("type").is_in(["R7", "R8"]) & (pl.col("side") == "R")).select("root_id", "type", "column_id", "p", "q")
r7 = ds[pr_cols.filter(pl.col("type") == "R7")["root_id"].to_list()]
r8 = ds[pr_cols.filter(pl.col("type") == "R8")["root_id"].to_list()]
pr = r7 | r8
print(pr, "| R7:", len(r7), "R8:", len(r8), "| columns:", pr_cols["column_id"].n_unique())
template = pr_cols.select("column_id", "p", "q").unique().sort("column_id")   # the right-eye hex template

def hex_dist(pq_a, pq_b):
    """Axial hex distance, broadcasting over (..., 2) arrays."""
    dp = pq_a[..., 0] - pq_b[..., 0]; dq = pq_a[..., 1] - pq_b[..., 1]
    return (np.abs(dp) + np.abs(dq) + np.abs(dp + dq)) / 2

def hex_map(values: pl.DataFrame, value_col: str, title: str, cmap="Viridis", size=13):
    """Colour each column of the right-eye template by a value (columns absent from `values` stay grey)."""
    t = template.join(values.select("column_id", value_col), on="column_id", how="left")
    pq = t.select("p", "q").to_numpy().astype(float)
    xy = pq @ (np.array([[np.sqrt(3) / 2, -np.sqrt(3) / 2], [0.5, 0.5]]).T * np.sqrt(3))
    v = t[value_col].to_numpy().astype(float)
    miss = np.isnan(v)
    fig = go.Figure()
    if miss.any():  # columns with no value stay grey
        fig.add_trace(go.Scatter(x=xy[miss, 0], y=xy[miss, 1], mode="markers", hoverinfo="skip", showlegend=False,
                                 marker=dict(symbol="hexagon2", size=size, color="#dddddd", line=dict(width=0.5, color="white"))))
    fig.add_trace(go.Scatter(x=xy[~miss, 0], y=xy[~miss, 1], mode="markers", showlegend=False, text=[f"column {c}: {x:.3g}" for c, x in zip(t["column_id"].to_numpy()[~miss], v[~miss])],
                             marker=dict(symbol="hexagon2", size=size, color=v[~miss], colorscale=cmap, showscale=True, colorbar=dict(title=value_col), line=dict(width=0.5, color="white"))))
    fig.update_layout(title=title, width=560, height=520, yaxis=dict(scaleanchor="x", scaleratio=1, visible=False), xaxis=dict(visible=False), margin=dict(l=10, r=10, t=40, b=10))
    return fig

hex_map(pr_cols.group_by("column_id").len().rename({"len": "photoreceptors"}), "photoreceptors", "R7 + R8 cells per column").show()

## 2.3 The two-hop pathway matrix

Two blocks of the connectivity give the whole pathway: `A_in` (photoreceptors × Dm9, synapses from each R7/R8 onto each Dm9) and `A_out` (Dm9 × photoreceptors). Their product is the synapse-weighted two-hop coupling, and the product of their boolean versions counts how many Dm9s link each pair. Rows and columns are root ids in `pr` order.

In [ ]:
A_in = ds.connectivity[pr, dm9].sparse.astype(np.float64)     # (n_pr, n_dm9)
A_out = ds.connectivity[dm9, pr].sparse.astype(np.float64)    # (n_dm9, n_pr)
print("A_in", A_in.shape, A_in.nnz, "pairs |", "A_out", A_out.shape, A_out.nnz, "pairs")
W = (A_in @ A_out).tocsr()                                    # synapse-weighted two-hop coupling
N_shared = ((A_in > 0).astype(np.int32) @ (A_out > 0).astype(np.int32)).tocsr()   # number of Dm9 intermediaries
counts = np.bincount(N_shared.data)[1:]
share = counts / counts.sum()
print("connected (source, target) pairs:", N_shared.nnz, "| coupled through 1, 2, 3, 4+ Dm9s:", [f"{s:.1%}" for s in share[:3]], f"{share[3:].sum():.2%}")
print("a photoreceptor reaches on average", f"{(N_shared > 0).sum(axis=1).mean():.1f}", "others through Dm9")

In [ ]:
# the same numbers split by photoreceptor type, as four labelled matrices
is_r7 = np.isin(pr.ids, r7.ids)
groups = {"R7": np.nonzero(is_r7)[0], "R8": np.nonzero(~is_r7)[0]}
rows = []
for a, ia in groups.items():
    for b, ib in groups.items():
        sub = N_shared[ia][:, ib]
        c = np.bincount(sub.data, minlength=5)[1:]
        rows.append({"pathway": f"{a} -> Dm9 -> {b}", "pairs": int(sub.nnz), "one Dm9": c[0] / sub.nnz, "two": c[1] / sub.nnz, "three": c[2] / sub.nnz, "mean synapse-weighted coupling": float(W[ia][:, ib].sum() / sub.nnz)})
pl.DataFrame(rows)

## 2.4 Receptive and projective fields

Per Dm9: the set of photoreceptors it receives from (RF) and sends to (PF), their sizes, and the Jaccard overlap between them, at the cell level and at the column level.

In [ ]:
def field(dm9_cell, direction):
    t = dm9_cell.inputs() if direction == "in" else dm9_cell.outputs()
    col = "pre" if direction == "in" else "post"
    return t.filter(pl.col("type").is_in(["R7", "R8"]))[col].to_list()

col_of = dict(zip(pr_cols["root_id"].to_list(), pr_cols["column_id"].to_list()))
rows = []
for c in dm9:
    rf, pf = set(field(c, "in")), set(field(c, "out"))
    rf_cols, pf_cols = {col_of[r] for r in rf if r in col_of}, {col_of[r] for r in pf if r in col_of}
    rows.append({"root_id": c.root_id, "rf_cells": len(rf), "pf_cells": len(pf), "rf_columns": len(rf_cols), "pf_columns": len(pf_cols),
                 "jaccard_cells": len(rf & pf) / max(len(rf | pf), 1), "jaccard_columns": len(rf_cols & pf_cols) / max(len(rf_cols | pf_cols), 1)})
fields = pl.DataFrame(rows)
display(fields.describe())
px.histogram(fields.to_pandas(), x="jaccard_columns", nbins=20, title="RF/PF overlap per Dm9 (column-level Jaccard)").show()

## 2.5 Distance decay and the offset kernel

Place every (source, target) pair on the grid, bin the two-hop coupling by hex distance, and average by offset (Δp, Δq) to get a kernel. Two versions matter:

- **masked**: only pairs that share a Dm9 ("given a shared Dm9, how strong is the coupling?");
- **unmasked**: all pairs, zeros included ("what is the expected coupling at this distance?"), which is what a network model needs.

Rows are normalized to their maximum first, so every source contributes on the same scale.

In [ ]:
pq = pr_cols.select("p", "q").to_numpy().astype(float)
order = np.searchsorted(np.sort(pr_cols["root_id"].to_numpy()), pr.ids)          # align pr_cols rows to pr order
pq = pq[np.argsort(pr_cols["root_id"].to_numpy())][order]
row_max = np.asarray(W.max(axis=1).todense()).ravel()
Wn = sp.diags(np.where(row_max > 0, 1 / np.maximum(row_max, 1e-12), 0)) @ W    # row-max normalized
Wn = Wn.tocsr()
D = hex_dist(pq[:, None, :], pq[None, :, :]).astype(int)                          # (n_pr, n_pr) hex distances

def decay(Wmat, D, masked):
    Wd = Wmat.toarray()
    d = D.ravel(); w = Wd.ravel()
    keep = (w > 0) if masked else np.ones_like(w, dtype=bool)
    d, w = d[keep], w[keep]
    cnt = np.bincount(d); s = np.bincount(d, weights=w)
    with np.errstate(invalid="ignore"):
        return np.arange(len(cnt)), s / cnt, cnt

fig = go.Figure()
for masked, name in ((True, "masked: pairs sharing a Dm9"), (False, "unmasked: all pairs")):
    d, mean, cnt = decay(Wn, D, masked)
    fig.add_trace(go.Scatter(x=d[:12], y=mean[:12], mode="lines+markers", name=name))
fig.update_layout(title="Two-hop coupling vs hex distance (row-max normalized)", xaxis_title="hex distance (columns)", yaxis_title="mean coupling", yaxis_type="log", height=420)
fig.show()
d, mean, cnt = decay(Wn, D, True)
half = d[np.nonzero(mean <= mean[0] / 2)[0][0]]
print("masked curve: coupling at d=1 is", f"{mean[1] / mean[0]:.0%}", "of d=0; half-maximum reached at d =", half)

In [ ]:
# offset kernel K(dp, dq): average normalized coupling over all pairs with the same offset (unmasked, min 10 pairs)
Wd = Wn.toarray()
dp = (pq[None, :, 0] - pq[:, None, 0]).astype(int).ravel(); dq = (pq[None, :, 1] - pq[:, None, 1]).astype(int).ravel()
k = pl.DataFrame({"dp": dp, "dq": dq, "w": Wd.ravel()}).group_by("dp", "dq").agg(pl.col("w").mean().alias("K"), pl.len().alias("n")).filter(pl.col("n") >= 10)
k = k.with_columns((pl.col("K") / k.filter((pl.col("dp") == 0) & (pl.col("dq") == 0))["K"].item()).alias("K"))
k = k.with_columns(pl.Series("r", hex_dist(k.select("dp", "dq").to_numpy().astype(float), np.zeros((k.height, 2)))))
radial = k.filter(pl.col("r") <= 6).group_by("r").agg(pl.col("K").mean()).sort("r")
print("radial profile of K (unmasked):", [f"{v:.3f}" for v in radial["K"]])
kk = k.filter(pl.col("r") <= 5).with_columns((pl.col("dp") * 1000 + pl.col("dq")).alias("column_id"), pl.col("dp").alias("p"), pl.col("dq").alias("q"))
template_backup = template
template = kk.select("column_id", "p", "q")
hex_map(kk, "K", "Offset kernel K(dp, dq), unmasked, normalized to K(0,0)", cmap="Hot", size=26).show()
template = template_backup

## 2.6 Coverage: how many Dm9s read from, and write to, each column

In [ ]:
def coverage(direction):
    seen = {}
    for c in dm9:
        for col in {col_of[r] for r in field(c, direction) if r in col_of}:
            seen[col] = seen.get(col, 0) + 1
    return pl.DataFrame({"column_id": list(seen), "n_dm9": list(seen.values())})

rf_cov, pf_cov = coverage("in"), coverage("out")
diff = rf_cov.join(pf_cov, on="column_id", how="full", coalesce=True, suffix="_pf").fill_null(0).with_columns((pl.col("n_dm9_pf") - pl.col("n_dm9")).alias("pf_minus_rf"))
hex_map(rf_cov, "n_dm9", "RF coverage: Dm9s receiving from each column").show()
hex_map(diff, "pf_minus_rf", "PF minus RF coverage", cmap="RdBu").show()
print("columns read by no Dm9:", template.height - rf_cov.height, "| written by no Dm9:", template.height - pf_cov.height)

## 2.7 Photoreceptors outside the Dm9 network, and within-column R7-R8 contacts

In [ ]:
in_deg = np.asarray((A_in > 0).sum(axis=1)).ravel(); out_deg = np.asarray((A_out > 0).sum(axis=0)).ravel()
print("photoreceptors with no input to any right Dm9:", int((in_deg == 0).sum()), "| receiving no Dm9 output:", int((out_deg == 0).sum()), "| neither:", int(((in_deg == 0) & (out_deg == 0)).sum()))

# direct R7 <-> R8 synapses inside the same column
B = ds.connectivity[r7, r8].long.join(pr_cols.select(pl.col("root_id").alias("pre"), pl.col("column_id").alias("c_pre")), on="pre").join(pr_cols.select(pl.col("root_id").alias("post"), pl.col("column_id").alias("c_post")), on="post")
same = B.filter(pl.col("c_pre") == pl.col("c_post"))
B2 = ds.connectivity[r8, r7].long.join(pr_cols.select(pl.col("root_id").alias("pre"), pl.col("column_id").alias("c_pre")), on="pre").join(pr_cols.select(pl.col("root_id").alias("post"), pl.col("column_id").alias("c_post")), on="post")
same2 = B2.filter(pl.col("c_pre") == pl.col("c_post"))
print(f"R7 -> R8 direct: {B.height} pairs, {same.height} within the same column (mean {same['n_syn'].mean():.1f} synapses)")
print(f"R8 -> R7 direct: {B2.height} pairs, {same2.height} within the same column (mean {same2['n_syn'].mean():.1f} synapses)")
print("columns with mutual R7 <-> R8 contact:", len(set(same['c_pre']) & set(same2['c_pre'])), "of", template.height)

## 2.8 From connectivity to physiology: one Dm9's synapses on its skeleton

The influence analysis proper injects current at every R7/R8 input synapse of every Dm9 and reads the voltage at its output synapses. The package provides the pieces: the synapse locations, the skeleton, compartments, `comp.nearest` to place synapses, and `Cable` to solve. Here is the setup for one representative Dm9 (median RF size); the population loop is the same code over `dm9`.

In [ ]:
median_rf = int(fields["rf_columns"].median())
rep = ds[int(fields.filter(pl.col("rf_columns") == median_rf)["root_id"][0])]
sk = rep.skeleton(); comp = cnx.morph.segment(sk); m = cnx.models.Cable(comp)
syn_in = rep.synapses("in").filter(pl.col("neuropil").is_not_null() | True)
pr_in = syn_in.join(pr_cols.select(pl.col("root_id").alias("pre"), "type", "column_id"), on="pre")     # R7/R8 -> this Dm9
comps = comp.nearest(cnx.xyz(pr_in) / 1e3)
print(rep, "|", len(comp), "compartments |", pr_in.height, "photoreceptor input synapses on", len(np.unique(comps)), "compartments")
# voltage at every compartment for one 10 pA injection per input synapse, averaged per source photoreceptor
Vsyn = np.stack([m.steady_state({int(c): 10e-12}) for c in comps])              # (n_syn, n_comp)
per_source = pr_in.with_columns(pl.Series("row", np.arange(pr_in.height))).group_by("pre", "type", "column_id").agg(pl.col("row"))
out_syn = rep.synapses("out").join(pr_cols.select(pl.col("root_id").alias("post"), pl.col("column_id").alias("column_out")), on="post")
out_comps = comp.nearest(cnx.xyz(out_syn) / 1e3)
src = per_source.row(0, named=True)
v = Vsyn[src["row"]].mean(axis=0)                                                 # per-synapse-normalized Dm9 voltage for that source
col_v = pl.DataFrame({"column_id": out_syn["column_out"], "V": v[out_comps]}).group_by("column_id").agg(pl.col("V").mean())
print(f"source {src['type']} in column {src['column_id']}: peak {v.max():.2f} mV on the Dm9; output columns reached: {col_v.height}")
hex_map(col_v, "V", f"Dm9 voltage at its output synapses when {src['type']} (column {src['column_id']}) is driven at 10 pA", cmap="Hot").show()

That last map is one row of the influence matrix; running it over every source photoreceptor and every Dm9 gives the full R7/R8 → Dm9 → R7/R8 influence matrices, and the decay, kernel and ellipse analyses of Part 2 apply to them unchanged.